In [ ]:
import unittest
from math import sqrt

from src.main.world_objects.robot_objects.position import Position


class TestCacheBehavior(unittest.TestCase):

    def setUp(self):
        # Clear the caches before each test to ensure isolation
        self.pos1 = Position(1.0, 2.0)
        self.pos2 = Position(4.0, 6.0)
        # Clear the Position's cache
        self.pos1.distance_to.cache_clear()

    def test_position_distance_cache_behavior(self):
        # Test cache behavior for Position's distance_to method
        distance1 = self.pos1.distance_to(self.pos2)
        self.assertEqual(distance1, sqrt((1.0 - 4.0) ** 2 + (2.0 - 6.0) ** 2))
        self.assertEqual(self.pos1.distance_to.cache_info().hits, 0)

        distance2 = self.pos1.distance_to(self.pos2)
        self.assertEqual(distance2, distance1)
        self.assertEqual(self.pos1.distance_to.cache_info().hits, 1)

        # Adding a new Position to fill the cache
        self.pos1.distance_to(Position(7.0, 8.0))
        self.assertEqual(self.pos1.distance_to.cache_info().currsize, 2)

        # Adding another new Position should evict the oldest entry
        self.pos1.distance_to(Position(9.0, 10.0))
        self.assertEqual(self.pos1.distance_to.cache_info().currsize, 3)

    def test_cache_clear(self):

        # Fill the cache for Position's distance_to
        self.pos1.distance_to(self.pos2)
        self.pos1.distance_to(Position(7.0, 8.0))
        self.assertEqual(self.pos1.distance_to.cache_info().currsize, 2)

        # Clear the cache
        self.pos1.distance_to.cache_clear()
        self.assertEqual(self.pos1.distance_to.cache_info().currsize, 0)


if __name__ == "__main__":
    unittest.main()


In [ ]:
import unittest
from math import sqrt

from src.main.world_objects.robot_objects.position import Position, InvalidPositionError
from src.main.world_objects.robot_objects.degrees import Degrees


def parameterized_test(test_cases):  #
    def decorator(test_func):  #
        def wrapper(self):  #
            for case in test_cases:  #
                with self.subTest(case=case):  #
                    test_func(self, *case)

        return wrapper  #

    return decorator  #


class TestPosition(unittest.TestCase):

    def test_initialization(self):
        """Test the initialization of Position instances with valid coordinates."""
        valid_positions = [  #
            (0, 0),
            (1.5, -2.3),
            (-1, 1),
            (3.14159, 2.71828),
        ]
        for x, y in valid_positions:
            with self.subTest(x=x, y=y):
                pos = Position(x, y)  #
                self.assertEqual(pos.x, x)
                self.assertEqual(pos.y, y)

    @parameterized_test(
        [
            (None, 1),
            (1, "string"),
            ([], {}),
            (float("nan"), 0),
            (0, float("inf")),
            (1, -float("inf")),
        ]
    )
    def test_initialization_with_invalid_coordinates(self, x, y):  #
        with self.assertRaises(InvalidPositionError):
            Position(x, y)  #

    def test_distance_to(self):
        """Test the distance calculation between two Position instances."""
        p1 = Position(0, 0)
        p2 = Position(3, 4)
        self.assertEqual(p1.distance_to(p2), 5)  # 3-4-5 triangle

        p3 = Position(1, 1)
        p4 = Position(4, 5)
        self.assertEqual(p3.distance_to(p4), 5)  # 3-4-5 triangle again

    def test_distance_to_invalid(self):
        """Test handling of invalid distance comparisons."""
        p1 = Position(0, 0)
        with self.assertRaises(ValueError):
            p1.distance_to("not a Position")

    def test_move(self):
        """Test moving the Position by a given angle and steps."""
        pos = Position(0, 0)
        angle = Degrees(90)
        moved_pos = pos.move(angle, 1)
        self.assertTrue(abs(moved_pos.x - 0) < 0.01)  # cos(90°) = 0
        self.assertTrue(abs(moved_pos.y - 1) < 0.01)  # sin(90°) = 1

        angle = Degrees(0)
        moved_pos = pos.move(angle, 1)  # 1,1
        self.assertTrue(abs(moved_pos.x - 1) < 0.01)  # cos(0°) = 1
        self.assertTrue(abs(moved_pos.y - 0) < 0.01)  # sin(0°) = 0

        angle = Degrees(45)
        moved_pos = pos.move(angle, sqrt(2))  # Should move to (1, 1)
        self.assertTrue(abs(moved_pos.x - 1) < 0.01)
        self.assertTrue(abs(moved_pos.y - 1) < 0.01)

    def test_move_invalid_angle(self):
        """Test moving with an invalid angle type."""
        pos = Position(0, 0)
        with self.assertRaises(ValueError):
            pos.move("not an angle", 1)  #

    def test_is_in(self):
        """Test if a Position is within a defined rectangular area."""
        p = Position(2, 3)
        top_left = Position(1, 4)
        bottom_right = Position(3, 2)

        self.assertTrue(p.is_in(top_left, bottom_right))

        p_outside = Position(4, 3)
        self.assertFalse(p_outside.is_in(top_left, bottom_right))

    def test_is_in_invalid_position(self):
        """Test is_in with invalid Position instances."""
        p = Position(2, 3)
        with self.assertRaises(ValueError):
            p.is_in("not a Position", Position(3, 2))  #

    def test_addition(self):
        """Test adding two Position instances."""
        p1 = Position(1, 2)
        p2 = Position(3, 4)
        result = p1 + p2
        self.assertEqual(result.x, 4)
        self.assertEqual(result.y, 6)

    def test_subtraction(self):
        """Test subtracting two Position instances."""
        p1 = Position(5, 5)
        p2 = Position(3, 3)
        result = p1 - p2
        self.assertEqual(result.x, 2)
        self.assertEqual(result.y, 2)

    def test_equality(self):
        """Test equality of Position instances."""
        p1 = Position(1, 2)
        p2 = Position(1, 2)
        p3 = Position(2, 1)
        self.assertEqual(p1, p2)
        self.assertNotEqual(p1, p3)

    def test_repr(self):
        """Test the __repr__ method."""
        p = Position(1.234567, 2.345678)
        self.assertEqual(repr(p), "Position(x=1.234567, y=2.345678)")

    def test_str(self):
        """Test the __str__ method."""
        p = Position(1.234567, 2.345678)
        self.assertEqual(str(p), "(1.234567, 2.345678)")

    def test_format_value(self):
        """Test the value formatting in the string representation."""
        self.assertEqual(Position._format_value(1.234567), "1.234567")  #
        self.assertEqual(Position._format_value(1.0), "1")  #
        self.assertEqual(Position._format_value(2.71828), "2.718280")  #


if __name__ == "__main__":
    unittest.main()


In [ ]:
import unittest
from math import radians, cos, sin, isclose

from src.main.world_objects.robot_objects.position import Position
from src.main.world_objects.robot_objects.degrees import Degrees


class TestPosition(unittest.TestCase):

    def setUp(self):
        self.start_pos = Position(0, 0)

    def test_initial_position(self):
        pos = Position(0, 0)
        self.assertEqual(pos.x, 0)
        self.assertEqual(pos.y, 0)

    def test_move_north(self):
        angle = Degrees(90)
        new_pos = self.start_pos.move(angle, 10)
        self.assertTrue(isclose(new_pos.x, 0, abs_tol=1e-9))
        self.assertTrue(isclose(new_pos.y, 10, abs_tol=1e-9))

    def test_move_south(self):
        angle = Degrees(270)
        new_pos = self.start_pos.move(angle, 10)
        self.assertTrue(isclose(new_pos.x, 0, abs_tol=1e-9))
        self.assertTrue(isclose(new_pos.y, -10, abs_tol=1e-9))

    def test_move_east(self):
        angle = Degrees(0)
        new_pos = self.start_pos.move(angle, 10)
        self.assertTrue(isclose(new_pos.x, 10, abs_tol=1e-9))
        self.assertTrue(isclose(new_pos.y, 0, abs_tol=1e-9))

    def test_move_west(self):
        angle = Degrees(180)
        new_pos = self.start_pos.move(angle, 10)
        self.assertTrue(isclose(new_pos.x, -10, abs_tol=1e-9))
        self.assertTrue(isclose(new_pos.y, 0, abs_tol=1e-9))

    def test_move_multiple_directions(self):
        pos = Position(1, 1)
        new_pos = pos.move(Degrees(90), 3).move(Degrees(0), 2)
        self.assertTrue(isclose(new_pos.x, 3, abs_tol=1e-9))
        self.assertTrue(isclose(new_pos.y, 4, abs_tol=1e-9))

    def test_move_zero_steps(self):
        pos = Position(5, -5)
        new_pos = pos.move(Degrees(90), 0)
        self.assertEqual(new_pos, pos)

    def test_move_large_steps(self):
        pos = Position(0, 0)
        new_pos = pos.move(Degrees(90), 1000000)
        self.assertTrue(isclose(new_pos.x, 0, abs_tol=1e-9))
        self.assertTrue(isclose(new_pos.y, 1000000, abs_tol=1e-9))

    def test_move_negative_steps(self):
        pos = Position(10, 10)
        new_pos = pos.move(Degrees(180), -5)
        self.assertTrue(isclose(new_pos.x, 15, abs_tol=1e-9))
        self.assertTrue(isclose(new_pos.y, 10, abs_tol=1e-9))

    def test_move_without_direction(self):
        # Move North (90 degrees) by 2 steps
        pos_after_north = self.start_pos.move(Degrees(90), 2)
        print(f"After moving North: {pos_after_north}")
        self.assertTrue(isclose(pos_after_north.x, 0, abs_tol=1e-6))
        self.assertTrue(isclose(pos_after_north.y, 2, abs_tol=1e-6))

        # Move West (180 degrees) by 1 step from the new position
        pos_after_west = pos_after_north.move(Degrees(180), 1)
        print(f"After moving West: {pos_after_west}")
        self.assertTrue(isclose(pos_after_west.x, -1, abs_tol=1e-6))
        self.assertTrue(isclose(pos_after_west.y, 2, abs_tol=1e-6))

        # Move South (270 degrees) by 2 steps from the new position
        pos_after_south = pos_after_west.move(Degrees(270), 2)
        print(f"After moving South: {pos_after_south}")
        self.assertTrue(isclose(pos_after_south.x, -1, abs_tol=1e-6))
        self.assertTrue(isclose(pos_after_south.y, 0, abs_tol=1e-6))

        # Move East (0 degrees) by 1 step from the new position
        pos_after_east = pos_after_south.move(Degrees(0), 1)
        print(f"After moving East: {pos_after_east}")
        self.assertTrue(isclose(pos_after_east.x, 0, abs_tol=1e-6))
        self.assertTrue(isclose(pos_after_east.y, 0, abs_tol=1e-6))
        #
        # # Check that the final position is the same as the starting position
        self.assertEqual(pos_after_east, self.start_pos)

    def test_angle_normalization(self):
        angle = Degrees(450)  # Equivalent to 90 degrees
        new_pos = self.start_pos.move(angle, 10)
        self.assertTrue(isclose(new_pos.x, 0, abs_tol=1e-9))
        self.assertTrue(isclose(new_pos.y, 10, abs_tol=1e-9))

    def test_negative_angle(self):
        angle = Degrees(-45)  # Equivalent to 315 degrees
        steps = 10
        radians(angle.angle)
        expected_x = steps * cos(radians(45))
        expected_y = -steps * sin(radians(45))
        new_pos = self.start_pos.move(angle, steps)
        self.assertTrue(isclose(new_pos.x, expected_x, abs_tol=1e-9))
        self.assertTrue(isclose(new_pos.y, expected_y, abs_tol=1e-9))


if __name__ == "__main__":
    unittest.main()


In [ ]:
import unittest
from src.main.world_objects.robot_objects.position import Position


class MyTestCase(unittest.TestCase):
    def test_eq_other_object(self):
        """Test inequality with a non-Position object."""
        self.assertFalse(Position(1, 2) == (1, 2))  # Compare with a tuple

    def test_eq_not_position(self):
        """Test inequality with an object of a different class."""
        self.assertFalse(Position(1, 2) == "Position(1, 2)")  # Compare with a string

    def test_same_coord_are_equal(self):
        """Test equality of a Position objects."""
        a = Position(1, 2)
        self.assertTrue(a == Position(1, 2))


if __name__ == "__main__":
    unittest.main()


In [ ]:
import unittest
from src.main.world_objects.robot_objects.position import Position


class MyTestCase(unittest.TestCase):
    def test_position_inside_bounds(self):
        top_left = Position(-100, 200)
        bottom_right = Position(100, -200)
        pos = Position(0, 0)
        self.assertTrue(
            pos.is_in(top_left, bottom_right),
            "Position (0, 0) should be inside the bounds",
        )

    def test_position_on_boundary(self):
        top_left = Position(-100, 200)
        bottom_right = Position(100, -200)
        pos = Position(-100, 200)
        self.assertTrue(
            pos.is_in(top_left, bottom_right),
            "Position (-100, 200) should be on the boundary and thus inside the bounds",
        )
        pos = Position(100, -200)
        self.assertTrue(
            pos.is_in(top_left, bottom_right),
            "Position (100, -200) should be on the boundary and thus inside the bounds",
        )
        pos = Position(-100, -200)
        self.assertTrue(
            pos.is_in(top_left, bottom_right),
            "Position (-100, -200) should be on the boundary and thus inside the bounds",
        )
        pos = Position(100, 200)
        self.assertTrue(
            pos.is_in(top_left, bottom_right),
            "Position (100, 200) should be on the boundary and thus inside the bounds",
        )

    def test_position_outside_bounds(self):
        top_left = Position(-100, 200)
        bottom_right = Position(100, -200)
        pos = Position(-101, 0)
        self.assertFalse(
            pos.is_in(top_left, bottom_right),
            "Position (-101, 0) should be outside the bounds",
        )
        pos = Position(0, 201)
        self.assertFalse(
            pos.is_in(top_left, bottom_right),
            "Position (0, 201) should be outside the bounds",
        )
        pos = Position(101, 0)
        self.assertFalse(
            pos.is_in(top_left, bottom_right),
            "Position (101, 0) should be outside the bounds",
        )
        pos = Position(0, -201)
        self.assertFalse(
            pos.is_in(top_left, bottom_right),
            "Position (0, -201) should be outside the bounds",
        )

    def test_position_edge_cases(self):
        top_left = Position(-100, 200)
        bottom_right = Position(100, -200)
        pos = Position(-100, 199)
        self.assertTrue(
            pos.is_in(top_left, bottom_right),
            "Position (-100, 199) should be inside the bounds",
        )
        pos = Position(100, -199)
        self.assertTrue(
            pos.is_in(top_left, bottom_right),
            "Position (100, -199) should be inside the bounds",
        )
        pos = Position(-99, 200)
        self.assertTrue(
            pos.is_in(top_left, bottom_right),
            "Position (-99, 200) should be inside the bounds",
        )
        pos = Position(99, -200)
        self.assertTrue(
            pos.is_in(top_left, bottom_right),
            "Position (99, -200) should be inside the bounds",
        )


if __name__ == "__main__":
    unittest.main()


In [ ]:
import unittest
from src.main.world_objects.robot_objects.position import Position


class TestPosition(unittest.TestCase):

    def setUp(self):
        """Set up test variables."""
        self.p1 = Position(1, 2)
        self.p2 = Position(3, 4)
        self.p3 = Position(1, 2)
        self.p4 = Position(5, 6)

    def test_initialization(self):
        """Test creation of Position objects."""
        self.assertEqual(self.p1.x, 1)
        self.assertEqual(self.p1.y, 2)

    def test_repr(self):
        """Test the __repr__ method."""
        self.assertEqual(repr(self.p2), "Position(x=3, y=4)")

    def test_str(self):
        """Test the __str__ method."""
        self.assertEqual(str(self.p4), "(5, 6)")

    def test_position_inequality(self):
        """Test that two Position objects with different values are not equal."""
        p_diff = Position(2, 3)
        self.assertNotEqual(self.p1, p_diff)

    def test_repr_not_null(self):
        """Ensure the __repr__ method does not return None."""
        self.assertIsNotNone(repr(self.p1))

    def test_str_not_null(self):
        """Ensure the __str__ method does not return None."""
        self.assertIsNotNone(str(self.p1))

    def test_eq_different_position(self):
        """Test inequality of two Position objects with different coordinates."""
        self.assertFalse(self.p1 == self.p2)

    def test_eq_same_position(self):
        """Test equality of two Position objects with the same coordinates."""
        a = Position(1, 2)
        b = Position(1, 2)
        self.assertTrue(a == b)


if __name__ == "__main__":
    unittest.main()


In [ ]:
import unittest
from src.main.world_objects.robot_objects.position import Position
from math import sqrt


class MyTestCase(unittest.TestCase):
    def test_distance_to_same_position(self):
        """Test distance to the same position (should be 0)."""
        p1 = Position(1, 2)
        self.assertEqual(p1.distance_to(Position(1, 2)), 0.0)

    def test_distance_to_different_position(self):
        """Test distance between two different positions."""
        p1 = Position(1, 2)
        p2 = Position(4, 6)
        expected_distance = sqrt((4 - 1) ** 2 + (6 - 2) ** 2)
        self.assertAlmostEqual(p1.distance_to(p2), expected_distance)

    def test_distance_to_origin(self):
        """Test distance from a position to the origin (0, 0)."""
        p = Position(3, 4)
        origin = Position(0, 0)
        expected_distance = sqrt(3**2 + 4**2)
        self.assertAlmostEqual(p.distance_to(origin), expected_distance)

    def test_distance_negative_coordinates(self):
        """Test distance between positions with negative coordinates."""
        p1 = Position(-1, -1)
        p2 = Position(3, 3)
        expected_distance = sqrt((3 - (-1)) ** 2 + (3 - (-1)) ** 2)
        self.assertAlmostEqual(p1.distance_to(p2), expected_distance)

    def test_distance_to_large_coordinates(self):
        """Test distance with large coordinate values."""
        p1 = Position(10000, 10000)
        p2 = Position(-10000, -10000)
        expected_distance = sqrt((10000 - (-10000)) ** 2 + (10000 - (-10000)) ** 2)
        self.assertAlmostEqual(p1.distance_to(p2), expected_distance)

    def test_distance_precision(self):
        """Test distance to verify precision with floating-point numbers."""
        p1 = Position(1.000001, 1.000001)
        p2 = Position(1.000002, 1.000002)
        expected_distance = sqrt(
            (1.000002 - 1.000001) ** 2 + (1.000002 - 1.000001) ** 2
        )
        self.assertAlmostEqual(p1.distance_to(p2), expected_distance, places=6)


if __name__ == "__main__":
    unittest.main()
